
# Phase 6 — Inventory Optimization

**Project:** Supply Chain Demand Forecasting & Inventory Optimization  
**Notebook:** `06_inventory_optimization.ipynb`

## Objective

Convert the 30-day demand forecasts from Phase 5 into interpretable inventory decisions:

**Forecast Demand → Lead-Time Demand → Demand Variability → Safety Stock → Reorder Point → Stockout Risk → Excess Inventory → Recommended Order Quantity → Priority / Action**

### Important methodological assumptions

- The dataset does **not** provide an observed supplier lead-time field.
- Lead time is therefore treated as an explicit scenario assumption rather than fabricated historical data.
- Base lead time: **7 days**
- Lead-time scenarios: **3, 7, and 14 days**
- Base service level: **95%**
- Service-level scenarios: **90%, 95%, and 99%**
- Recommendations are business rules derived from documented calculations, not model predictions.
- Forecast bias and uncertainty from Phase 5 must be considered when interpreting inventory recommendations.


In [ ]:

# ============================================================
# 1. Configuration
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import norm

warnings.filterwarnings("ignore")

SALES_PATH = "/content/drive/MyDrive/SupplychainManagement/sales_data.csv"
FORECAST_PATH = "/content/drive/MyDrive/SupplychainManagement/final_demand_forecasts.csv"

OUTPUT_DIR = "/content/drive/MyDrive/SupplychainManagement/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

INVENTORY_OUTPUT_PATH = os.path.join(
    OUTPUT_DIR, "inventory_recommendations.csv"
)

VALIDATION_OUTPUT_PATH = os.path.join(
    OUTPUT_DIR, "inventory_validation_report.csv"
)

LEAD_TIME_SCENARIOS = [3, 7, 14]
SERVICE_LEVEL_SCENARIOS = [0.90, 0.95, 0.99]

BASE_LEAD_TIME = 7
BASE_SERVICE_LEVEL = 0.95

print("Configuration loaded successfully.")
print("Sales file:", SALES_PATH)
print("Forecast file:", FORECAST_PATH)


## 2. Load Source Data

In [ ]:

# ============================================================
# 2. Load source files
# ============================================================

if not os.path.exists(SALES_PATH):
    raise FileNotFoundError(f"Sales file not found: {SALES_PATH}")

if not os.path.exists(FORECAST_PATH):
    raise FileNotFoundError(f"Forecast file not found: {FORECAST_PATH}")

sales = pd.read_csv(SALES_PATH)
forecasts = pd.read_csv(FORECAST_PATH)

print("Sales shape:", sales.shape)
print("Forecast shape:", forecasts.shape)

print("\nSales columns:")
print(sales.columns.tolist())

print("\nForecast columns:")
print(forecasts.columns.tolist())


## 3. Inspect and Standardize Column Names

The notebook does not assume exact capitalization or spacing. It attempts to identify the required fields from the actual files and stops with a clear error if a required field cannot be identified.

In [ ]:

# ============================================================
# 3. Column detection helpers
# ============================================================

def normalize_name(x):
    return (
        str(x)
        .strip()
        .lower()
        .replace("_", " ")
        .replace("-", " ")
    )

def find_column(df, candidates, required=True):
    normalized = {normalize_name(c): c for c in df.columns}

    for candidate in candidates:
        key = normalize_name(candidate)
        if key in normalized:
            return normalized[key]

    # Fuzzy containment fallback
    for col in df.columns:
        ncol = normalize_name(col)
        for candidate in candidates:
            ncandidate = normalize_name(candidate)
            if ncandidate in ncol or ncol in ncandidate:
                return col

    if required:
        raise KeyError(
            f"Could not identify required column from candidates: {candidates}\n"
            f"Available columns: {df.columns.tolist()}"
        )
    return None


sales_date_col = find_column(
    sales, ["Date", "date", "Order Date", "Sales Date"]
)

sales_store_col = find_column(
    sales, ["Store ID", "Store", "Store_ID", "store_id"]
)

sales_product_col = find_column(
    sales, ["Product ID", "Product", "Product_ID", "product_id", "SKU"]
)

inventory_col = find_column(
    sales, ["Inventory Level", "Inventory", "Stock", "Current Inventory"]
)

units_sold_col = find_column(
    sales, ["Units Sold", "Units_Sold", "Sales Quantity", "Quantity Sold"]
)

units_ordered_col = find_column(
    sales, ["Units Ordered", "Units_Ordered", "Order Quantity", "Quantity Ordered"]
)

demand_col = find_column(
    sales, ["Demand", "Demand Quantity", "Demand Qty"]
)

forecast_date_col = find_column(
    forecasts, ["Date", "date", "Forecast Date"]
)

forecast_store_col = find_column(
    forecasts, ["Store ID", "Store", "Store_ID", "store_id"]
)

forecast_product_col = find_column(
    forecasts, ["Product ID", "Product", "Product_ID", "product_id", "SKU"]
)

forecast_demand_col = find_column(
    forecasts, ["Forecast Demand", "Forecast_Demand", "Forecast", "Predicted Demand", "prediction"]
)

print("Detected sales columns:")
print({
    "date": sales_date_col,
    "store": sales_store_col,
    "product": sales_product_col,
    "inventory": inventory_col,
    "units_sold": units_sold_col,
    "units_ordered": units_ordered_col,
    "demand": demand_col
})

print("\nDetected forecast columns:")
print({
    "date": forecast_date_col,
    "store": forecast_store_col,
    "product": forecast_product_col,
    "forecast_demand": forecast_demand_col
})


## 4. Build Standardized Working Tables

In [ ]:

# ============================================================
# 4. Standardize working tables
# ============================================================

hist = sales[
    [
        sales_date_col,
        sales_store_col,
        sales_product_col,
        inventory_col,
        units_sold_col,
        units_ordered_col,
        demand_col
    ]
].copy()

hist.columns = [
    "Date",
    "Store ID",
    "Product ID",
    "Inventory Level",
    "Units Sold",
    "Units Ordered",
    "Demand"
]

fcst = forecasts[
    [
        forecast_date_col,
        forecast_store_col,
        forecast_product_col,
        forecast_demand_col
    ]
].copy()

fcst.columns = [
    "Date",
    "Store ID",
    "Product ID",
    "Forecast Demand"
]

hist["Date"] = pd.to_datetime(hist["Date"], errors="coerce")
fcst["Date"] = pd.to_datetime(fcst["Date"], errors="coerce")

for col in ["Inventory Level", "Units Sold", "Units Ordered", "Demand"]:
    hist[col] = pd.to_numeric(hist[col], errors="coerce")

fcst["Forecast Demand"] = pd.to_numeric(
    fcst["Forecast Demand"], errors="coerce"
)

hist["Store ID"] = hist["Store ID"].astype(str).str.strip()
hist["Product ID"] = hist["Product ID"].astype(str).str.strip()

fcst["Store ID"] = fcst["Store ID"].astype(str).str.strip()
fcst["Product ID"] = fcst["Product ID"].astype(str).str.strip()

print("Standardized historical data:", hist.shape)
print("Standardized forecast data:", fcst.shape)


## 5. Step 1 — Inventory Data Validation

In [ ]:

# ============================================================
# 5. Step 1 — Inventory data validation
# ============================================================

required_inventory_cols = [
    "Inventory Level",
    "Units Sold",
    "Units Ordered",
    "Demand"
]

validation_rows = []

for col in required_inventory_cols:
    s = hist[col]

    validation_rows.append({
        "Variable": col,
        "Rows": len(s),
        "Missing Count": int(s.isna().sum()),
        "Missing %": round(float(s.isna().mean() * 100), 3),
        "Negative Count": int((s < 0).sum()),
        "Zero Count": int((s == 0).sum()),
        "Mean": s.mean(),
        "Median": s.median(),
        "Std Dev": s.std(),
        "Min": s.min(),
        "Max": s.max(),
        "P01": s.quantile(0.01),
        "P25": s.quantile(0.25),
        "P75": s.quantile(0.75),
        "P99": s.quantile(0.99)
    })

validation_report = pd.DataFrame(validation_rows)

display(validation_report)

validation_report.to_csv(
    VALIDATION_OUTPUT_PATH,
    index=False
)

print("\nValidation report saved to:")
print(VALIDATION_OUTPUT_PATH)


In [ ]:

# ============================================================
# 5A. Structural and temporal validation
# ============================================================

print("Duplicate historical rows:", hist.duplicated().sum())
print("Duplicate forecast rows:", fcst.duplicated().sum())

print("\nInvalid historical dates:", hist["Date"].isna().sum())
print("Invalid forecast dates:", fcst["Date"].isna().sum())

print("\nHistorical date range:")
print(hist["Date"].min(), "to", hist["Date"].max())

print("\nForecast date range:")
print(fcst["Date"].min(), "to", fcst["Date"].max())

print("\nHistorical Store × Product series:",
      hist[["Store ID", "Product ID"]].drop_duplicates().shape[0])

print("Forecast Store × Product series:",
      fcst[["Store ID", "Product ID"]].drop_duplicates().shape[0])

negative_summary = (
    hist[required_inventory_cols] < 0
).sum().sort_values(ascending=False)

print("\nNegative-value counts:")
display(negative_summary.to_frame("Count"))

zero_summary = (
    hist[required_inventory_cols] == 0
).sum().sort_values(ascending=False)

print("\nZero-value counts:")
display(zero_summary.to_frame("Count"))



### Validation interpretation

Negative values are **flagged, not automatically deleted**. A negative value can indicate a data-quality issue, adjustment, return, or another business process. The correct treatment depends on the dataset semantics.

Likewise, extreme values are investigated rather than removed merely to improve the optimization output.


In [ ]:

# ============================================================
# 5B. Missing-data and negative-data decision flags
# ============================================================

quality_flags = {
    "missing_inventory": int(hist["Inventory Level"].isna().sum()),
    "missing_units_sold": int(hist["Units Sold"].isna().sum()),
    "missing_units_ordered": int(hist["Units Ordered"].isna().sum()),
    "missing_demand": int(hist["Demand"].isna().sum()),
    "negative_inventory": int((hist["Inventory Level"] < 0).sum()),
    "negative_units_sold": int((hist["Units Sold"] < 0).sum()),
    "negative_units_ordered": int((hist["Units Ordered"] < 0).sum()),
    "negative_demand": int((hist["Demand"] < 0).sum()),
}

print("Quality flags:")
for k, v in quality_flags.items():
    print(f"{k}: {v}")

print("\nNo rows are deleted by this validation step.")


## 6. Historical Demand Variability

Demand variability is calculated at the **Store × Product** level from historical observations.

In [ ]:

# ============================================================
# 6. Demand variability by Store × Product
# ============================================================

# Exclude invalid demand values from statistical variability calculations.
# Do not silently rewrite the original data.
demand_valid = hist.loc[
    hist["Demand"].notna() & (hist["Demand"] >= 0)
].copy()

variability = (
    demand_valid
    .groupby(["Store ID", "Product ID"], as_index=False)
    .agg(
        Historical_Demand_Mean=("Demand", "mean"),
        Historical_Demand_Std=("Demand", "std"),
        Historical_Demand_Count=("Demand", "count"),
        Historical_Demand_Min=("Demand", "min"),
        Historical_Demand_Max=("Demand", "max")
    )
)

variability["Historical_Demand_Std"] = (
    variability["Historical_Demand_Std"].fillna(0)
)

variability["Coefficient_of_Variation"] = np.where(
    variability["Historical_Demand_Mean"] > 0,
    variability["Historical_Demand_Std"]
    / variability["Historical_Demand_Mean"],
    np.nan
)

print("Store × Product variability table:", variability.shape)
display(variability.head())


## 7. Determine Current Inventory

For each Store × Product series, current inventory is taken from the **latest valid historical observation**. This avoids treating the historical average inventory as today's inventory.

In [ ]:

# ============================================================
# 7. Current inventory snapshot
# ============================================================

inventory_valid = hist.loc[
    hist["Date"].notna() &
    hist["Inventory Level"].notna() &
    (hist["Inventory Level"] >= 0)
].copy()

latest_inventory = (
    inventory_valid
    .sort_values(["Store ID", "Product ID", "Date"])
    .groupby(["Store ID", "Product ID"], as_index=False)
    .tail(1)
    [["Store ID", "Product ID", "Date", "Inventory Level"]]
    .rename(columns={
        "Date": "Inventory Snapshot Date",
        "Inventory Level": "Current Inventory"
    })
)

print("Current inventory snapshots:", latest_inventory.shape)
display(latest_inventory.head())


## 8. Validate Phase 5 Forecasts

In [ ]:

# ============================================================
# 8. Forecast validation
# ============================================================

print("Missing forecast demand:", fcst["Forecast Demand"].isna().sum())
print("Negative forecast demand:", (fcst["Forecast Demand"] < 0).sum())

forecast_summary = fcst["Forecast Demand"].describe()
display(forecast_summary.to_frame("Forecast Demand"))

forecast_series = (
    fcst.groupby(["Store ID", "Product ID"])
    .size()
    .describe()
)

print("Forecast observations per Store × Product:")
display(forecast_series.to_frame("Count"))


In [ ]:

# ============================================================
# 8A. Confirm forecast horizon per Store × Product
# ============================================================

forecast_horizon = (
    fcst.groupby(["Store ID", "Product ID"])
    .agg(
        Forecast_Start=("Date", "min"),
        Forecast_End=("Date", "max"),
        Forecast_Days=("Date", "nunique")
    )
    .reset_index()
)

display(forecast_horizon["Forecast_Days"].describe().to_frame("Forecast Days"))

print("Series with exactly 30 unique forecast dates:",
      (forecast_horizon["Forecast_Days"] == 30).sum())

print("Total Store × Product forecast series:",
      len(forecast_horizon))


## 9. Align Historical Inventory, Variability, and Forecasts

In [ ]:

# ============================================================
# 9. Build the Store × Product optimization base
# ============================================================

series_base = (
    latest_inventory
    .merge(
        variability,
        on=["Store ID", "Product ID"],
        how="left"
    )
)

series_base = series_base.merge(
    forecast_horizon,
    on=["Store ID", "Product ID"],
    how="left"
)

print("Optimization base shape:", series_base.shape)

missing_variability = series_base["Historical_Demand_Std"].isna().sum()
missing_inventory = series_base["Current Inventory"].isna().sum()

print("Series missing demand variability:", missing_variability)
print("Series missing current inventory:", missing_inventory)



## 10. Forecast-Based Lead-Time Demand

For each Store × Product:

**Lead-Time Demand = sum of forecast demand during the selected lead-time period**

The calculation is performed directly from the dated Phase 5 forecasts.

For the base scenario:

**Lead-Time Demand = first 7 forecast days summed**

This is preferable to multiplying a historical average by lead time because Phase 5 already produced date-specific future demand forecasts.


In [ ]:

# ============================================================
# 10. Base 7-day forecast lead-time demand
# ============================================================

fcst_sorted = fcst.sort_values(
    ["Store ID", "Product ID", "Date"]
).copy()

fcst_sorted["Forecast Day Number"] = (
    fcst_sorted
    .groupby(["Store ID", "Product ID"])
    .cumcount() + 1
)

base_ltd = (
    fcst_sorted.loc[
        fcst_sorted["Forecast Day Number"] <= BASE_LEAD_TIME
    ]
    .groupby(["Store ID", "Product ID"], as_index=False)
    .agg(
        Lead_Time_Demand=("Forecast Demand", "sum")
    )
)

print("Base 7-day lead-time demand:")
display(base_ltd.head())


## 11. Safety Stock

For this scenario model, safety stock is calculated using historical daily demand variability:

**Safety Stock = Z × σ × √Lead Time**

where:
- `Z` = service-level z-score
- `σ` = historical daily demand standard deviation
- `Lead Time` = assumed replenishment lead time in days

In [ ]:

# ============================================================
# 11. Safety stock for base scenario
# ============================================================

base_z = norm.ppf(BASE_SERVICE_LEVEL)

series_base["Z_Value"] = base_z

series_base["Safety Stock"] = (
    series_base["Z_Value"]
    * series_base["Historical_Demand_Std"]
    * np.sqrt(BASE_LEAD_TIME)
)

series_base["Safety Stock"] = series_base["Safety Stock"].clip(lower=0)

print(f"Base service level: {BASE_SERVICE_LEVEL:.0%}")
print(f"Z value: {base_z:.4f}")
display(
    series_base[
        [
            "Store ID",
            "Product ID",
            "Historical_Demand_Std",
            "Safety Stock"
        ]
    ].head()
)


## 12. Reorder Point

**Reorder Point = Lead-Time Demand + Safety Stock**

In [ ]:

# ============================================================
# 12. Base reorder point
# ============================================================

decision_base = series_base.merge(
    base_ltd,
    on=["Store ID", "Product ID"],
    how="left"
)

decision_base["Reorder Point"] = (
    decision_base["Lead_Time_Demand"]
    + decision_base["Safety Stock"]
)

decision_base["Inventory Gap"] = (
    decision_base["Reorder Point"]
    - decision_base["Current Inventory"]
)

display(
    decision_base[
        [
            "Store ID",
            "Product ID",
            "Current Inventory",
            "Lead_Time_Demand",
            "Safety Stock",
            "Reorder Point",
            "Inventory Gap"
        ]
    ].head(10)
)


## 13. Stockout Risk

Business-rule classification:

- **High:** Current Inventory < Reorder Point
- **Medium:** Current Inventory is at or above the reorder point but within 20% of it
- **Low:** Current Inventory is more than 20% above the reorder point

These are decision rules, not statistical probabilities.

In [ ]:

# ============================================================
# 13. Stockout risk classification
# ============================================================

decision_base["Inventory_to_ROP_Ratio"] = np.where(
    decision_base["Reorder Point"] > 0,
    decision_base["Current Inventory"] / decision_base["Reorder Point"],
    np.nan
)

def classify_risk(row):
    inventory = row["Current Inventory"]
    rop = row["Reorder Point"]

    if pd.isna(inventory) or pd.isna(rop):
        return "Unknown"

    if inventory < rop:
        return "High"

    if inventory <= 1.20 * rop:
        return "Medium"

    return "Low"

decision_base["Risk"] = decision_base.apply(
    classify_risk,
    axis=1
)

print("Risk distribution:")
display(
    decision_base["Risk"]
    .value_counts(dropna=False)
    .rename_axis("Risk")
    .to_frame("Count")
)


## 14. Excess Inventory

To avoid an arbitrary label, excess inventory is defined relative to a documented target level:

**Target Inventory = Lead-Time Demand + Safety Stock**

For this Phase 6 decision layer, inventory materially above the target is flagged when:

**Current Inventory > 1.50 × Reorder Point**

This is a business-rule threshold and should be interpreted as a screening signal, not a measured holding-cost calculation.

In [ ]:

# ============================================================
# 14. Excess inventory screening
# ============================================================

EXCESS_MULTIPLIER = 1.50

decision_base["Excess_Inventory_Units"] = (
    decision_base["Current Inventory"]
    - decision_base["Reorder Point"]
).clip(lower=0)

decision_base["Excess_Inventory_Flag"] = np.where(
    decision_base["Current Inventory"]
    > EXCESS_MULTIPLIER * decision_base["Reorder Point"],
    "Yes",
    "No"
)

print("Excess inventory flags:")
display(
    decision_base["Excess_Inventory_Flag"]
    .value_counts()
    .rename_axis("Excess Inventory")
    .to_frame("Count")
)


## 15. Recommended Order Quantity

For replenishment, the initial target is the reorder point:

**Recommended Order Quantity = max(Reorder Point − Current Inventory, 0)**

This is intentionally a simple, interpretable policy. It does not pretend to optimize economic order quantity because the dataset does not provide the cost parameters needed for a defensible EOQ calculation.

In [ ]:

# ============================================================
# 15. Recommended order quantity
# ============================================================

decision_base["Recommended Order Quantity"] = (
    decision_base["Reorder Point"]
    - decision_base["Current Inventory"]
).clip(lower=0)

decision_base["Recommended Order Quantity"] = np.ceil(
    decision_base["Recommended Order Quantity"]
)

display(
    decision_base[
        [
            "Store ID",
            "Product ID",
            "Current Inventory",
            "Reorder Point",
            "Recommended Order Quantity"
        ]
    ].head(10)
)


## 16. Recommended Action and Priority

In [ ]:

# ============================================================
# 16. Action rules
# ============================================================

def action_rule(row):
    risk = row["Risk"]
    excess = row["Excess_Inventory_Flag"]
    qty = row["Recommended Order Quantity"]

    if risk == "High":
        return "Replenish"

    if excess == "Yes":
        return "Review Excess"

    if risk == "Medium":
        return "Monitor"

    if qty > 0:
        return "Plan Replenishment"

    return "Maintain"

decision_base["Recommended Action"] = decision_base.apply(
    action_rule,
    axis=1
)

priority_map = {
    "High": 1,
    "Medium": 2,
    "Low": 3,
    "Unknown": 4
}

decision_base["Priority"] = (
    decision_base["Risk"].map(priority_map).fillna(4).astype(int)
)

decision_base = decision_base.sort_values(
    ["Priority", "Recommended Order Quantity"],
    ascending=[True, False]
)

display(
    decision_base[
        [
            "Store ID",
            "Product ID",
            "Current Inventory",
            "Lead_Time_Demand",
            "Safety Stock",
            "Reorder Point",
            "Inventory Gap",
            "Risk",
            "Recommended Order Quantity",
            "Recommended Action",
            "Priority"
        ]
    ].head(20)
)


## 17. Lead-Time Scenario Analysis

In [ ]:

# ============================================================
# 17. Lead-time scenarios
# ============================================================

scenario_results = []

for lead_time in LEAD_TIME_SCENARIOS:
    temp_fcst = fcst_sorted.loc[
        fcst_sorted["Forecast Day Number"] <= lead_time
    ]

    ltd = (
        temp_fcst
        .groupby(["Store ID", "Product ID"], as_index=False)
        .agg(
            Lead_Time_Demand=("Forecast Demand", "sum")
        )
    )

    temp = series_base[
        [
            "Store ID",
            "Product ID",
            "Current Inventory",
            "Historical_Demand_Std"
        ]
    ].merge(
        ltd,
        on=["Store ID", "Product ID"],
        how="left"
    )

    for service_level in SERVICE_LEVEL_SCENARIOS:
        z = norm.ppf(service_level)

        temp["Safety Stock"] = (
            z
            * temp["Historical_Demand_Std"]
            * np.sqrt(lead_time)
        ).clip(lower=0)

        temp["Reorder Point"] = (
            temp["Lead_Time_Demand"]
            + temp["Safety Stock"]
        )

        temp["Recommended Order Quantity"] = np.ceil(
            (
                temp["Reorder Point"]
                - temp["Current Inventory"]
            ).clip(lower=0)
        )

        temp["Inventory Gap"] = (
            temp["Reorder Point"]
            - temp["Current Inventory"]
        )

        temp["Lead Time"] = lead_time
        temp["Service Level"] = service_level

        scenario_results.append(
            temp[
                [
                    "Store ID",
                    "Product ID",
                    "Current Inventory",
                    "Lead Time",
                    "Service Level",
                    "Lead_Time_Demand",
                    "Historical_Demand_Std",
                    "Safety Stock",
                    "Reorder Point",
                    "Inventory Gap",
                    "Recommended Order Quantity"
                ]
            ]
        )

scenario_df = pd.concat(
    scenario_results,
    ignore_index=True
)

print("Scenario table:", scenario_df.shape)
display(scenario_df.head(10))


## 18. Phase 5 Forecast-Bias Risk

The Phase 5 analysis reported systematic underprediction during high-demand observations. Therefore, the inventory recommendations should not be interpreted as if forecasts were perfectly calibrated.

The optimization layer does **not silently inflate the forecast** to compensate. Instead, forecast bias is explicitly reported as a limitation and risk factor.

In [ ]:

# ============================================================
# 18. Forecast risk diagnostic
# ============================================================

print("Forecast-risk interpretation:")
print("- Inventory recommendations depend directly on Phase 5 forecasts.")
print("- Any systematic forecast underprediction can reduce estimated replenishment needs.")
print("- High-demand periods therefore require additional review.")
print("- No undocumented forecast uplift is applied in this notebook.")


## 19. Final Recommendation Dataset

In [ ]:

# ============================================================
# 19. Final output table
# ============================================================

final_output = decision_base.copy()

final_output["Lead Time"] = BASE_LEAD_TIME
final_output["Service Level"] = BASE_SERVICE_LEVEL

final_output = final_output.rename(columns={
    "Lead_Time_Demand": "Lead Time Demand",
    "Historical_Demand_Std": "Demand Std Dev"
})

final_columns = [
    "Inventory Snapshot Date",
    "Store ID",
    "Product ID",
    "Forecast_Start",
    "Forecast_End",
    "Forecast_Days",
    "Current Inventory",
    "Lead Time",
    "Lead Time Demand",
    "Demand Std Dev",
    "Service Level",
    "Z_Value",
    "Safety Stock",
    "Reorder Point",
    "Inventory Gap",
    "Excess_Inventory_Units",
    "Excess_Inventory_Flag",
    "Recommended Order Quantity",
    "Risk",
    "Recommended Action",
    "Priority"
]

final_output = final_output[
    [c for c in final_columns if c in final_output.columns]
].copy()

final_output.to_csv(
    INVENTORY_OUTPUT_PATH,
    index=False
)

print("Final inventory recommendation file created:")
print(INVENTORY_OUTPUT_PATH)

print("\nFinal shape:", final_output.shape)
display(final_output.head(20))


## 20. Final Validation

In [ ]:

# ============================================================
# 20. Final validation checks
# ============================================================

checks = {
    "No negative recommended order quantities":
        (final_output["Recommended Order Quantity"] >= 0).all(),

    "No negative safety stock":
        (final_output["Safety Stock"] >= 0).all(),

    "No negative reorder points":
        (final_output["Reorder Point"] >= 0).all(),

    "Required recommendation columns present":
        set([
            "Store ID",
            "Product ID",
            "Current Inventory",
            "Lead Time",
            "Lead Time Demand",
            "Demand Std Dev",
            "Service Level",
            "Safety Stock",
            "Reorder Point",
            "Recommended Order Quantity",
            "Risk",
            "Recommended Action"
        ]).issubset(final_output.columns)
}

for check, result in checks.items():
    print(f"{check}: {'PASS' if result else 'FAIL'}")

print("\nMissing values in final output:")
display(
    final_output.isna()
    .sum()
    .sort_values(ascending=False)
    .rename("Missing Count")
    .to_frame()
)


## 21. Summary Statistics

In [ ]:

# ============================================================
# 21. Business summary
# ============================================================

print("=== INVENTORY OPTIMIZATION SUMMARY ===")

print("\nRisk:")
display(
    final_output["Risk"]
    .value_counts()
    .rename_axis("Risk")
    .to_frame("Count")
)

print("\nRecommended actions:")
display(
    final_output["Recommended Action"]
    .value_counts()
    .rename_axis("Action")
    .to_frame("Count")
)

print("\nTotal recommended order quantity:",
      final_output["Recommended Order Quantity"].sum())

print("Total current inventory:",
      final_output["Current Inventory"].sum())

print("Total safety stock:",
      final_output["Safety Stock"].sum())

print("High-risk series:",
      (final_output["Risk"] == "High").sum())

print("Excess-inventory flagged series:",
      (final_output["Excess_Inventory_Flag"] == "Yes").sum())



## 22. Interpretation and Limitations

### What this analysis provides

The notebook converts Phase 5 forecast output into an interpretable inventory decision layer using:

1. Forecast-based lead-time demand.
2. Historical Store × Product demand variability.
3. Explicit service-level assumptions.
4. Explicit lead-time assumptions.
5. Reorder points.
6. Stockout-risk screening.
7. Excess-inventory screening.
8. Recommended replenishment quantities.

### Important limitations

- Lead time is assumed because the dataset does not contain observed supplier lead time.
- Safety stock uses historical demand standard deviation and therefore does not fully model forecast-error uncertainty.
- The recommendation policy does not include ordering costs, holding costs, shortage costs, minimum order quantities, or supplier constraints.
- Forecast underprediction from Phase 5 can cause inventory requirements to be underestimated.
- Risk categories are business rules, not probabilities of stockout.
- Excess inventory is a screening rule rather than a financial holding-cost calculation.
- No financial savings or business impact is claimed.

These limitations are consistent with the project's requirement to document assumptions and avoid unsupported business-impact claims.



# Phase 6 Conclusion

The inventory optimization layer successfully translates the Phase 5 demand forecasts into operational inventory metrics and recommendations.

The resulting `inventory_recommendations.csv` provides a traceable Store × Product decision table containing current inventory, lead-time demand, demand variability, safety stock, reorder point, inventory gap, recommended order quantity, risk, and recommended action.

The next phase is **Phase 7 — Power BI**, where these outputs can be presented through the Inventory Decision Center alongside demand and forecast-performance views.
